# 모형 노트북 템플릿 (학습 → OOB 부트스트랩 → 커트라인 → 시각화)

복사해서 `02_logit`, `03_xgb`, `04_xgb_mono`를 만들고 **1번 칸의 `MODEL_KIND`만** 바꿉니다.
- 설정값은 `00_config`, 전처리는 `lib_prep`에서 옵니다. 이 노트북에 숫자나 전처리 코드를 직접 쓰지 않습니다.
- 2번 칸(공통 함수)은 각자 고치지 않습니다. 나중에 `lib_engine.ipynb`(logit 담당)로 옮깁니다.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !curl -sL https://raw.githubusercontent.com/Walnutbrown/kdt14th-lendingclub/main/colab_setup.py -o /content/colab_setup.py
    %run -i /content/colab_setup.py
# 1. 설정
MODEL_KIND = "xgb"        # "logit" | "xgb" | "xgb_mono"
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb

import time, joblib
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
OUT = OUT_DIR / MODEL_KIND
(OUT / "models_b").mkdir(parents=True, exist_ok=True)   # 출력 형식은 02_logit(logit 담당)과 같음 (2026-11-26 통일)

In [ ]:
# 2. 공통 함수 (개별 수정 금지. lib_engine으로 옮길 예정)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from xgboost import XGBClassifier


def make_oob_splits(n_samples: int, n_bootstrap: int = 100, seed: int = 42) -> list[tuple[np.ndarray, np.ndarray]]:
    """Out-of-Bag(OOB) 부트스트랩 위치 인덱스 생성 (Train_b 복원추출 N / Val_b 미추출 ~36.8%, 겹침 0건)."""
    rng = np.random.default_rng(seed)
    all_idx = np.arange(n_samples, dtype=np.int64)
    splits: list[tuple[np.ndarray, np.ndarray]] = []
    for _ in range(n_bootstrap):
        tr = rng.integers(0, n_samples, size=n_samples, dtype=np.int64)
        in_tr = np.zeros(n_samples, dtype=bool)
        in_tr[tr] = True
        splits.append((tr, all_idx[~in_tr]))
    return splits


# lib_engine.split_fingerprint와 같은 함수 (②③도 같은 값이 나와야 06_collect 비교가 맞음)
def split_fingerprint(splits, n_samples=None):
    """분할 지문: B회 OOB 인덱스(oob_idx)를 회차 순서대로 이어 붙인 int64 배열의 sha256 앞 12자 (팀 Notion 명세).
    ①②③이 같은 n, B, seed로 분할했으면 같은 값이 나온다. n_samples는 받기만 하고 쓰지 않는다(호출 형식 통일용).
    summary.json의 split_hash로 남기고, 06_collect가 세 모형의 값이 같은지 비교한다."""
    import hashlib
    oob_all = np.concatenate([np.asarray(oob, dtype=np.int64) for _, oob in splits])
    return hashlib.sha256(np.ascontiguousarray(oob_all).tobytes()).hexdigest()[:12]


def fit_model(X, y, mono=None):
    if MODEL_KIND == "logit":
        return LogisticRegression(**LOGIT_PARAMS).fit(X, y)
    params = {**XGB_FIXED, **XGB_PARAMS}
    if MODEL_KIND == "xgb_mono":
        params["monotone_constraints"] = mono
    return XGBClassifier(**params).fit(X, y)


def sharpe(r):
    s = r.std(ddof=1)
    return r.mean() / s if s > 0 else np.nan


def sr_curve(p, ret, rates=RATES, mode=SR_MODE):
    """p가 낮은 순으로 승인율 a만큼 승인했을 때의 SR과 확률 커트라인 τ."""
    order = np.argsort(p, kind="stable")
    p_s, r_s, n = p[order], ret[order], len(p)
    rows = []
    for a in rates:
        k = max(int(round(a * n)), 2)
        if mode == "approved_only":
            sr = sharpe(r_s[:k])
        elif mode == "with_treasury":
            sr = sharpe(np.concatenate([r_s[:k], np.zeros(n - k)]))
        else:
            raise ValueError(mode)
        rows.append((a, sr, p_s[k - 1]))
    return pd.DataFrame(rows, columns=["rate", "sr", "tau"])


def pick_cutoff(curve, min_rate=MIN_RATE):
    c = curve[curve.rate >= min_rate]
    row = c.loc[c.sr.idxmax()]
    return float(row.rate), float(row.tau), float(row.sr)

In [ ]:
# 3. 데이터 (01_prep 결과)
# Dev 입력 내용 지문: 학습에 쓴 parquet 파일의 sha256. 06_collect가 ①②③이 같은 Dev 버전을 썼는지 비교 (행 수가 같아도 내용이 다르면 잡힘)
import hashlib
DATA_FILES = {n: hashlib.sha256((DATA_DIR / n).read_bytes()).hexdigest() for n in ("dev_X_raw.parquet", "dev_meta.parquet")}
X = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet")
meta = pd.read_parquet(DATA_DIR / "dev_meta.parquet")
assert DATA_FILES == {n: hashlib.sha256((DATA_DIR / n).read_bytes()).hexdigest() for n in DATA_FILES}, "읽는 중에 Dev parquet이 바뀜 → 다시 실행"
y, ret = meta["y"].to_numpy(), meta["excess_ret"].to_numpy()
CATS = load_categories()
assert len(X) == len(meta)
print(X.shape, f"부도율 {y.mean():.4f}")

In [ ]:
# 4. OOB 부트스트랩: 매 회차 전처리와 모형을 Train_b에서만 학습
rows, curves = [], []
t0 = time.time()
splits = make_oob_splits(len(X), B, seed=SEED)
SPLIT_HASH = split_fingerprint(splits, len(X))   # 분할 지문 (summary.json의 split_hash)
for b, (tr, oob) in enumerate(splits):
    prep = make_preprocessor(MODEL_KIND, CATS).fit(X.iloc[tr])
    check_columns(prep, MODEL_KIND)
    Xtr = prep.transform(X.iloc[tr])
    model = fit_model(Xtr, y[tr], mono_constraints(prep))
    p_oob = model.predict_proba(prep.transform(X.iloc[oob]))[:, 1]

    curve = sr_curve(p_oob, ret[oob]).assign(b=b)
    a_b, tau_b, sr_b = pick_cutoff(curve)
    rows.append(dict(b=b, a_star=a_b, tau_star=tau_b, sr_max=sr_b,
                     auc=roc_auc_score(y[oob], p_oob), brier=brier_score_loss(y[oob], p_oob)))
    curves.append(curve)
    joblib.dump((prep, model), OUT / "models_b" / f"model_b{b:03d}.joblib")   # 6-C 안정성용

res = pd.DataFrame(rows)
curves = pd.concat(curves, ignore_index=True)
print(f"{B}회, {time.time() - t0:.0f}초 | 분할 지문 {SPLIT_HASH}")
res.head()

In [ ]:
# 5. 요약: 승인율 a*와 확률 τ* 중 어느 쪽이 회차마다 덜 흔들리나
cv = lambda s: s.std(ddof=1) / s.mean()
pd.Series({"a* 중앙값": res.a_star.median(), "τ* 중앙값": res.tau_star.median(),
           "a* 변동계수": cv(res.a_star), "τ* 변동계수": cv(res.tau_star),
           "SR 최대 중앙값": res.sr_max.median(), "AUC 중앙값": res.auc.median()}).round(4)

In [ ]:
# 6. 시각화
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for _, g in curves.groupby("b"):
    ax[0].plot(g.rate, g.sr, color="0.7", lw=1)
mean_c = curves.groupby("rate").sr.mean()
ax[0].plot(mean_c.index, mean_c.values, color="C0", lw=2.5, label="회차 평균")
ax[0].axvline(res.a_star.median(), color="C3", ls="--", label=f"a* 중앙값 {res.a_star.median():.2f}")
ax[0].set(title=f"OOB SR 곡선 ({MODEL_KIND}, B={B}, {SR_MODE}, {STAGE})", xlabel="승인율", ylabel="SR")
ax[0].legend()
ax[1].hist(res.a_star, bins=15, color="C0"); ax[1].set(title="회차별 a*_b", xlabel="승인율")
ax[2].hist(res.tau_star, bins=15, color="C1"); ax[2].set(title="회차별 τ*_b", xlabel="부도확률 커트라인")
plt.tight_layout()
fig.savefig(OUT / "bootstrap_cutoff.png", dpi=300)
plt.show()

In [ ]:
# 7. Dev 100% 재학습과 저장 (Test는 열지 않음)
prep_full = make_preprocessor(MODEL_KIND, CATS).fit(X)
check_columns(prep_full, MODEL_KIND)
X_full = prep_full.transform(X)
final = fit_model(X_full, y, mono_constraints(prep_full))
joblib.dump((prep_full, final), OUT / "final_model.joblib")
res[["b", "a_star", "tau_star", "sr_max", "auc", "brier"]].to_csv(OUT / "bootstrap_results.csv", index=False)
curves[["b", "rate", "sr", "tau"]].to_csv(OUT / "bootstrap_curves.csv", index=False)
# summary.json 공통 키 (①②③ 같음, 05_test가 읽음): model, a_star, tau_star_median, columns, config_hash, lib_prep_hash
# split_hash·n_samples: 분할 지문. 06_collect가 ①②③이 같은 분할을 썼는지 비교
# files: 같은 실행에서 방금 쓴 결과 CSV의 sha256. 06_collect가 읽은 CSV와 대조 (summary는 항상 CSV 다음에 마지막으로 씀)
import hashlib
RESULT_FILES = {n: hashlib.sha256((OUT / n).read_bytes()).hexdigest() for n in ("bootstrap_results.csv", "bootstrap_curves.csv")}
summary = dict(model=MODEL_KIND, a_star=float(res.a_star.median()),
               tau_star_median=float(res.tau_star.median()), B=B, seed=SEED, split_hash=SPLIT_HASH, n_samples=len(X), data_files=DATA_FILES, files=RESULT_FILES, sr_mode=SR_MODE, min_rate=MIN_RATE,
               hyperparameters=(LOGIT_PARAMS if MODEL_KIND == "logit" else {**XGB_FIXED, **XGB_PARAMS}),
               columns=list(X_full.columns), config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH,
               meta=run_meta())
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=str))
print("저장:", OUT.relative_to(ROOT), "| config", summary["config_hash"], "| lib_prep", summary["lib_prep_hash"], "| split", summary["split_hash"])